In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import os
os.environ['CUDA_VISIBLE_DEVICES'] = '0'
os.environ["XLA_PYTHON_CLIENT_PREALLOCATE"]="false"

import pathlib
from copy import deepcopy
from functools import partial

import time
from tqdm import tqdm
import numpy as np
import matplotlib.pyplot as plt
import matplotlib as mpl
mpl.rcParams['text.usetex'] = True
mpl.rcParams['text.latex.preamble']=r"\usepackage{bm}"

In [ ]:
import jax
import jax.numpy as jnp
import equinox as eqx
import diffrax

In [ ]:
from dmpe.utils.density_estimation import build_grid, DensityEstimate
from dmpe.utils.env_utils.fluid_tank_utils import setup_env as setup_fluid_tank_env
from dmpe.utils.env_utils.pendulum_utils import setup_env as setup_pendulum_env
from dmpe.utils.env_utils.cart_pole_utils import setup_env as setup_cart_pole_env
from dmpe.models.model_utils import simulate_ahead_with_env

---

---

In [ ]:
from admpe.evaluation.test_trajectories.simple_test_trajectories import SIGNALS

In [ ]:
signal_function = SIGNALS["aprbs"][0]
signal_params = SIGNALS["aprbs"][1]

In [ ]:
for setup in [setup_fluid_tank_env, setup_pendulum_env, setup_cart_pole_env]:
    env, penalty_function, featurize, env_params = setup()
    key = jax.random.PRNGKey(1234)
    signal_key, state_key = jax.random.split(key)
    n_steps = 500
    
    actions = 1.0 * signal_function(signal_key, n_steps, env.action_dim, **signal_params)
    
    initial_state = env.init_state(state_key, deterministic_state=False)
    initial_observation = env.generate_observation(initial_state)
    
    observation_list = []
    noise_levels = [0.0, *np.arange(1e-4, 1.1e-2, 1e-4).tolist()]
    
    for noise_variance in noise_levels:
        test_env = eqx.tree_at(lambda e: e.process_noise_variance, env, noise_variance)
    
        observations, _ = simulate_ahead_with_env(test_env, initial_observation, initial_state, actions)
        observation_list.append(observations)
    
    fig, axs = plt.subplots(env.obs_dim + env.action_dim, figsize=(12, 5), sharex=True, constrained_layout=True)
    for o in observation_list:
        for i in range(env.obs_dim):
            axs[i].plot(o[:, i])
    
    for ax in axs:
        ax.grid(True, alpha=0.3)
    axs[-1].plot(actions)
    plt.show()

---

In [ ]:
for setup in [setup_fluid_tank_env, setup_pendulum_env, setup_cart_pole_env]:
    env, penalty_function, featurize, env_params = setup()
    key = jax.random.PRNGKey(44)
    signal_key, state_key = jax.random.split(key)
    n_steps = 500
    
    actions = 1.0 * signal_function(signal_key, n_steps, env.action_dim, **signal_params)
    
    initial_state = env.init_state(state_key, deterministic_state=False)
    initial_observation = env.generate_observation(initial_state)
    
    observation_list = []
    noise_levels = [0.0, 1e-4]#1e-6, 1e-5, 1e-4]#, 1e-3]#, 1e-2]
    
    for noise_variance in noise_levels:
        test_env = eqx.tree_at(lambda e: e.process_noise_variance, env, noise_variance)
    
        observations, _ = simulate_ahead_with_env(test_env, initial_observation, initial_state, actions)
        observation_list.append(observations)
    
    fig, axs = plt.subplots(env.obs_dim + env.action_dim, figsize=(12, 5), sharex=True, constrained_layout=True)
    for o in observation_list:
        for i in range(env.obs_dim):
            axs[i].plot(o[:, i])
    
    for ax in axs:
        ax.grid(True, alpha=0.3)
    axs[-1].plot(actions)
    plt.show()